## 2.3 Integration: Adding `Primary_Diagnosis_Desc`

### What I am doing
I add a new column called `Primary_Diagnosis_Desc` to the main dataset. For each encounter, the column shows the plain-English description of its primary diagnosis code (`diag_1`), using the descriptions I scraped in step 2.2.

### How I label the rows
- If `diag_1` is one of the top 20 codes, it gets the scraped description (for example, `428` becomes "Heart failure").
- If `diag_1` is any other code, or is missing, it gets the label **"Not in Top 20"**, as the project specification asks.
- If a top 20 code was found in the data but the scraper could not find its description, it gets **"Description Unavailable"**. This way a scraping problem is never mixed up with a rare diagnosis.

### Expected outcome
The same dataset with one extra column. No rows should be added or removed, and the new column should have no empty values.

In [ ]:
# 2.3 Integration: create Primary_Diagnosis_Desc

rows_before = len(df_cleaned)

desc_map = dict(zip(icd9_lookup['diag_1'], icd9_lookup['icd9_description']))


def label_primary_diagnosis(code):
    """Return the description for a top 20 code, otherwise 'Not in Top 20'."""
    if code not in desc_map:
        return "Not in Top 20"
    description = desc_map[code]
    return description if pd.notna(description) else "Description Unavailable"


df_cleaned['Primary_Diagnosis_Desc'] = df_cleaned['diag_1'].apply(label_primary_diagnosis)

assert len(df_cleaned) == rows_before, "Row count changed!"
assert df_cleaned['Primary_Diagnosis_Desc'].isna().sum() == 0, "Empty values found!"

print("Primary_Diagnosis_Desc added successfully.")
print(f"Dataset shape: {df_cleaned.shape}\n")
print(df_cleaned['Primary_Diagnosis_Desc'].value_counts().head(10))

df_cleaned[['encounter_id', 'diag_1', 'Primary_Diagnosis_Desc']].head(10)

### Findings from Step 2.3
- The new column `Primary_Diagnosis_Desc` was added to the dataset. The number of rows stayed the same (**100,111**) and there are no empty values.
- **"Not in Top 20"** is the largest group with **49,274** encounters (about 49% of the data), because the top 20 codes cover only part of all the diagnoses.
- Among the described diagnoses, the most common is **Heart failure** with **6,735** encounters, followed by **Other forms of chronic ischemic heart disease** (6,554) and **Symptoms involving respiratory system and other chest symptoms** (4,016).

**Conclusion:** The cryptic ICD-9 codes can now be read as disease names, so they can be used in the analysis and the final report.